![ngo_project_image](ngo_project_image.jpg)

GoodThought NGO has been a catalyst for positive change, focusing its efforts on education, healthcare, and sustainable development to make a significant difference in communities worldwide. With this mission, GoodThought has orchestrated an array of assignments aimed at uplifting underprivileged populations and fostering long-term growth.

This project offers a hands-on opportunity to explore how data-driven insights can direct and enhance these humanitarian efforts. In this project, you'll engage with the GoodThought PostgreSQL database, which encapsulates detailed records of assignments, funding, impacts, and donor activities from 2010 to 2023. This comprehensive dataset includes:

- **`Assignments`:** Details about each project, including its name, duration (start and end dates), budget, geographical region, and the impact score.
- **`Donations`:** Records of financial contributions, linked to specific donors and assignments, highlighting how financial support is allocated and utilized.
- **`Donors`:** Information on individuals and organizations that fund GoodThought’s projects, including donor types.

Refer to the below ERD diagram for a visual representation of the relationships between these data tables:
<img src="erd.png" alt="ERD" width="50%" height="50%">


You will execute SQL queries to answer two questions, as listed in the instructions. Good luck!


In [4]:
-- highest_donation_assignments
WITH donations_ranked AS (
    SELECT
        dnrs.donor_type,
        dntn.assignment_id,
        ROUND(SUM(dntn.amount), 2) AS rounded_total_donation_amount,
        ROW_NUMBER() OVER (
            --PARTITION BY dnrs.donor_type
            ORDER BY SUM(dntn.amount) DESC
        ) as Rank
    FROM
        public.donations dntn
	LEFT JOIN public.donors dnrs
		ON dntn.donor_id = dnrs.donor_id 
    GROUP BY
        dnrs.donor_type,
        dntn.assignment_id
)

SELECT 
	asmt.assignment_name,
	asmt.region,
	donations_ranked.rounded_total_donation_amount,
	donations_ranked.donor_type
FROM donations_ranked
LEFT JOIN public.assignments asmt
	ON donations_ranked.assignment_id = asmt.assignment_id
WHERE donations_ranked.rank < 6
ORDER BY rounded_total_donation_amount DESC;

,assignment_name,region,rounded_total_donation_amount,donor_type
0,Assignment_3033,East,3840.66,Individual
1,Assignment_300,West,3133.98,Organization
2,Assignment_4114,North,2778.57,Organization
3,Assignment_1765,West,2626.98,Organization
4,Assignment_268,East,2488.69,Individual


In [5]:
-- top_regional_impact_assignments
WITH donations_grouped AS (
    SELECT
        assignment_id,
        COUNT(donation_id) AS num_total_donations
    FROM
        public.donations
    GROUP BY
        assignment_id
),
assignments_ranked AS (
	SELECT
		asmt.assignment_id,
		asmt.assignment_name,
		asmt.region,
		asmt.impact_score,
		ROW_NUMBER() OVER (
			PARTITION BY asmt.region
			ORDER BY asmt.impact_score DESC
		) as rank
		FROM public.assignments asmt
)

SELECT 
	assignments_ranked.assignment_name,
	assignments_ranked.region,
	assignments_ranked.impact_score,
	donations_grouped.num_total_donations
FROM assignments_ranked
LEFT JOIN donations_grouped
	USING(assignment_id)
WHERE assignments_ranked.rank = 1
ORDER BY region ASC

,assignment_name,region,impact_score,num_total_donations
0,Assignment_316,East,10.00,2
1,Assignment_2253,North,9.99,1
2,Assignment_3547,South,10.00,1
3,Assignment_3764,West,9.99,1
